<a href="https://colab.research.google.com/github/Karidan/Intellectual-Data-Analysis-2025/blob/main/Lab4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. КОНФІГУРАЦІЯ


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import re
import os
import zipfile


2. ПІДГОТОВКА ДАНИХ


In [ ]:
BATCH_SIZE = 64
EPOCHS = 30
LATENT_DIM = 256
EMBEDDING_DIM = 128
NUM_SAMPLES = 15000

MAX_VOCAB_EN = 10000
MAX_VOCAB_UK = 10000


In [ ]:
DATA_PATH = "ukr.txt"

def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zа-яіїєґ0-9\s.,!?']", "", text)
    return text


In [ ]:
print("Завантаження даних...")

input_texts = []
target_texts = []

with open(DATA_PATH, encoding="utf-8", errors="ignore") as f:
    lines = f.read().split("\n")

for line in lines[:NUM_SAMPLES]:
    parts = line.split("\t")
    if len(parts) >= 2:
        en = clean_text(parts[0])
        uk = clean_text(parts[1])

        input_texts.append(en)
        target_texts.append("start_ " + uk + " _end")

print("Пар речень:", len(input_texts))


Завантаження даних...
Пар речень: 15000


In [ ]:
input_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_EN,
    oov_token="<unk>"
)
input_tokenizer.fit_on_texts(input_texts)

target_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_UK,
    filters="",
    oov_token="<unk>"
)
target_tokenizer.fit_on_texts(target_texts)


In [ ]:
encoder_seq = input_tokenizer.texts_to_sequences(input_texts)
decoder_seq = target_tokenizer.texts_to_sequences(target_texts)

max_encoder_len = max(len(s) for s in encoder_seq)
max_decoder_len = max(len(s) for s in decoder_seq)


In [ ]:
encoder_input = pad_sequences(
    encoder_seq,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input = pad_sequences(
    decoder_seq,
    maxlen=max_decoder_len,
    padding="post"
)


In [ ]:
decoder_target = decoder_input[:, 1:]
decoder_input = decoder_input[:, :-1]


In [ ]:
num_encoder_tokens = min(MAX_VOCAB_EN, len(input_tokenizer.word_index) + 1)
num_decoder_tokens = min(MAX_VOCAB_UK, len(target_tokenizer.word_index) + 1)


In [ ]:
# ---------- ENCODER ----------
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(num_encoder_tokens, EMBEDDING_DIM, mask_zero=True)(encoder_inputs)

encoder_lstm = LSTM(LATENT_DIM, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]


In [ ]:
# ---------- DECODER ----------
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(num_decoder_tokens, EMBEDDING_DIM, mask_zero=True)(decoder_inputs)

decoder_lstm = LSTM(
    LATENT_DIM,
    return_sequences=True,
    return_state=True
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)


In [ ]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │    345,728 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  1,066,240 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  2,140,810 │ lstm_1[0][0]      │
│                     │ 8330)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,341,258 (16.56 MB)

 Trainable params: 4,341,258 (16.56 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint("s2s_best.keras", save_best_only=True)
]

model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
    callbacks=callbacks
)


Epoch 1/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 248s 1s/step - accuracy: 0.4375 - loss: 5.9980 - val_accuracy: 0.2457 - val_loss: 4.1577
Epoch 2/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 238s 1s/step - accuracy: 0.2426 - loss: 3.5282 - val_accuracy: 0.2601 - val_loss: 3.9572
Epoch 3/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 227s 1s/step - accuracy: 0.2567 - loss: 3.1588 - val_accuracy: 0.2745 - val_loss: 3.8522
Epoch 4/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 239s 1s/step - accuracy: 0.2666 - loss: 2.8913 - val_accuracy: 0.2787 - val_loss: 3.7890
Epoch 5/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 270s 1s/step - accuracy: 0.2720 - loss: 2.6688 - val_accuracy: 0.2838 - val_loss: 3.7487
Epoch 6/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.2774 - loss: 2.4633 - val_accuracy: 0.2869 - val_loss: 3.7043
Epoch 7/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.2855 - loss: 2.2481 - val_accuracy: 0.2889 - val_loss: 3.7108
Epoch 8/30
188/188 ━━━━━━━━━━━━━━━━━━━━ 241s 1s/step - accuracy: 0.2929 - loss: 2.0503 - val_accu

In [29]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_h = Input(shape=(LATENT_DIM,))
decoder_state_c = Input(shape=(LATENT_DIM,))
decoder_states_inputs = [decoder_state_h, decoder_state_c]

dec_emb2 = model.layers[3](decoder_inputs)
decoder_outputs2, h2, c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2, h2, c2]
)


In [30]:
reverse_target_index = {v: k for k, v in target_tokenizer.word_index.items()}


In [31]:
def translate(sentence):
    sentence = clean_text(sentence)
    seq = input_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_encoder_len, padding="post")

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.array([[target_tokenizer.word_index["start_"]]])
    result = []

    for _ in range(max_decoder_len):
        output, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        token = np.argmax(output[0, -1, :])

        word = reverse_target_index.get(token)
        if word == "_end" or word is None:
            break

        result.append(word)
        target_seq = np.array([[token]])
        states = [h, c]

    return " ".join(result)


In [36]:
print("\n--- ТЕСТ ПЕРЕКЛАДУ ---")
for i in range(100):
    print("EN:", input_texts[i])
    print("GT:", target_texts[i])
    print("PR:", translate(input_texts[i]))
    print("-")



--- ТЕСТ ПЕРЕКЛАДУ ---
EN: go.
GT: start_ йди. _end
PR: будь реалістом.
-
EN: hi.
GT: start_ вітаю! _end
PR: привіт.
-
EN: hi.
GT: start_ привіт. _end
PR: привіт.
-
EN: hi.
GT: start_ привіт! _end
PR: привіт.
-
EN: run!
GT: start_ біжіть! _end
PR: привіт.
-
EN: run!
GT: start_ тікайте! _end
PR: привіт.
-
EN: run!
GT: start_ біжи! _end
PR: привіт.
-
EN: wow!
GT: start_ оце так! _end
PR: фантастика!
-
EN: wow!
GT: start_ клас! _end
PR: фантастика!
-
EN: wow!
GT: start_ класно! _end
PR: фантастика!
-
EN: wow!
GT: start_ ого! _end
PR: фантастика!
-
EN: wow!
GT: start_ ух ти! _end
PR: фантастика!
-
EN: fire!
GT: start_ пожежа! _end
PR: класно!
-
EN: help!
GT: start_ допоможіть! _end
PR: дякую помирають.
-
EN: jump!
GT: start_ стрибайте. _end
PR: видихай.
-
EN: jump.
GT: start_ стрибай. _end
PR: видихай.
-
EN: jump.
GT: start_ стрибайте. _end
PR: видихай.
-
EN: stop!
GT: start_ стій! _end
PR: фантастика!
-
EN: wait!
GT: start_ почекай! _end
PR: ласкаво просимо
-
EN: wait.
GT: start_ зачекай

Висновок до лабораторної роботи

У межах даної лабораторної роботи було реалізовано та досліджено нейронну модель машинного перекладу типу Seq2Seq (Encoder–Decoder) на основі LSTM для перекладу коротких англійських речень українською мовою. Було виконано повний цикл роботи з моделлю: підготовка та очищення корпусу паралельних текстів, токенізація, побудова архітектури моделі, навчання, а також тестування як на тренувальних прикладах, так і на нових фразах.

У процесі навчання модель продемонструвала стабільне зменшення функції втрат (loss) та зростання точності (accuracy) на тренувальній вибірці. Разом з тим, значення val_loss після певної кількості епох перестало покращуватись, що свідчить про обмежену здатність моделі до узагальнення та часткове перенавчання. Це є типовим результатом для класичної Seq2Seq-архітектури без механізму уваги (attention), особливо при роботі з багатозначними та короткими фразами.

Результати тестового перекладу показали такі характерні особливості:

модель коректно засвоїла найпоширеніші відповідності (наприклад, hi → привіт, exhale → видихай, listen → послухай);

для багатьох коротких і неоднозначних англійських фраз модель схильна віддавати найчастотніший або семантично “безпечний” переклад, ігноруючи контекст (наприклад, run! → привіт, wait! → ласкаво просимо);

у деяких випадках спостерігається семантична невідповідність, коли модель підставляє граматично коректну, але змістовно далеку фразу (help! → дякую помирають, i ran → я звільняюся).

Попри зазначені обмеження, модель успішно навчилася базовим відповідностям між мовами та підтвердила працездатність класичного підходу Seq2Seq для задач машинного перекладу. Отримані результати є задовільними для навчальної роботи та наочно демонструють як сильні сторони, так і недоліки даної архітектури.